In [1]:
from sklearn import svm

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy.stats as stats
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler as  StandardScaler
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, balanced_accuracy_score
from sklearn.model_selection import KFold, cross_validate, cross_val_score, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_selection import SelectKBest, f_classif, SelectFromModel
from imblearn.pipeline import Pipeline as im_Pipeline
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.metrics import roc_curve, confusion_matrix

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

In [2]:
df1 = pd.read_csv("master_tripel_cramps_dataset.csv")

df_ogmenstrual = df1[df1["phase"] == "Menstrual"].copy()

df_clean_menstrual_og = df_ogmenstrual.dropna(subset=[
    "cramps",
    "cramps binary",
    "cramps group",
    "phase",
    "lh",
    "estrogen",
    "nightly_temperature",
    "hr_mean",
    "glucose_median",
    "glucose_std"
]).copy()

cramps_map_menstrual = {
    "Not at all": 0,
    "Very Low/Little": 1,
    "Low": 2,
    "Moderate": 3,
    "High": 4,
    "Very High": 5
}

cramps_map_binary_menstrual = {
    "Low Pain": 0,
    "High Pain": 1,
}

cramps_map_group_menstrual = {
    "No Pain": 0,
    "Low Pain": 1,
    "High Pain": 2,
}

df_clean_menstrual_og["cramps"] = df_clean_menstrual_og["cramps"].map(cramps_map_menstrual)
df_clean_menstrual_og["cramps binary"] = df_clean_menstrual_og["cramps binary"].map(cramps_map_binary_menstrual)
df_clean_menstrual_og["cramps group"] = df_clean_menstrual_og["cramps group"].map(cramps_map_group_menstrual)

df_clean_menstrual_og = df_clean_menstrual_og.dropna(subset=["cramps binary"]).copy()
drop_cols = [
    "id",
    "phase",
    "cramps",
    "cramps binary",
    "cramps group",
    "study_interval_x",
    "study_interval_y",
    "day_in_study"
]

X = df_clean_menstrual_og.drop(columns=drop_cols)
y = df_clean_menstrual_og["cramps binary"].astype(int)

variables_num = X.columns.tolist()

In [3]:
groups = df_clean_menstrual_og["id"]

cv_outer = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_inner = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

import pickle 
with open("outer_splits_menstrual_binary.pkl", "rb") as f:
    outer_splits = pickle.load(f)

def get_best_rho(y_train, y_train_proba_oof, rhos=np.linspace(0.1, 0.9, 17)):

    best_rho = None
    best_f1 = -1

    for rho in rhos:

        y_pred = (y_train_proba_oof >= rho).astype(int)

        f1 = f1_score(y_train, y_pred, average="macro")

        if f1 > best_f1:
            best_f1 = f1
            best_rho = rho

    return best_rho

In [4]:
param_grid = {"KNN": {
                    "classifier__n_neighbors": [3, 5, 7, 9, 11],
                    "classifier__weights": ["uniform", "distance"],
                    "classifier__metric": ["euclidean", "manhattan"],
                    "feature_selection__k": [5, 10, 15, 20, "all"]}, 
              "LogReg_Youden": {
                    "classifier__C": [0.001, 0.01, 0.1, 1, 10],
                    "classifier__class_weight": ["balanced"],
                    "classifier__penalty": ["l1", "l2"],
                    "feature_selection__k": [5, 10, 15, 20, "all"]},
              "LogReg_Rho": {
                    "classifier__C": [0.001, 0.01, 0.1, 1, 10],
                    "classifier__class_weight": ["balanced"],
                    "classifier__penalty": ["l1", "l2"],
                    "feature_selection__k": [5, 10, 15, 20, "all"]},
              "Random_Forest": {
                    "classifier__n_estimators": [10, 20, 50, 100, 200, 250, 300, 500],
                    "classifier__max_depth": [1, 2, 3, 5, 7, 10],
                    "classifier__min_samples_leaf": [1, 5, 10],
                    "classifier__max_features": ["sqrt", "log2"],
                    "classifier__max_leaf_nodes": [2, 4, 8, 16, 32],
                    "classifier__class_weight": ["balanced"]},
              "SVM":{
                    "classifier__kernel": ["linear","rbf"],
                    "classifier__gamma": ["scale", "auto"],
                    "classifier__C": [0.01, 0.1, 1, 10],
                    "classifier__class_weight": ["balanced"],
                    "feature_selection__k": [5, 10, 15, 20, "all"]}

    }

In [10]:
model_df = []
type_response_df = []
accuracy_model = []
balanced_accuracy_model = []
f1_macro_model = []
f1_weighted_model = []
AUC_model = []
models = ["KNN", "LogReg_Youden", "LogReg_Rho","Random_Forest","SVM"]


accuracy = np.nan * np.ones((len(models), len(outer_splits)))
balanced_accuracy = np.nan * np.ones((len(models), len(outer_splits)))
f1_macro = np.nan * np.ones((len(models), len(outer_splits)))
f1_weighted = np.nan * np.ones((len(models), len(outer_splits)))
auc = np.nan * np.ones((len(models), len(outer_splits)))

best_params_all = {model: [] for model in models}
selected_features_all = {model: [] for model in models}
thresholds_lr_youden = []
thresholds_lr_rho = []

for j, (train_idx, test_idx) in enumerate(outer_splits):
    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    groups_train = groups.iloc[train_idx]
        
    for (i,model) in enumerate(models):
        preprocessor = ColumnTransformer(
            transformers=[
                ("zscore", StandardScaler(), variables_num)
        ])
        match model:
            case "LogReg_Youden":
                mdl = LogisticRegression( solver = "saga", max_iter=1000, random_state = 42)
            case "LogReg_Rho":
                mdl = LogisticRegression( solver = "saga", max_iter=1000, random_state = 42)    
            case "Random_Forest":
                mdl = RandomForestClassifier(random_state = 42)
            case "KNN":
                mdl = KNeighborsClassifier()
            case "SVM":
                mdl = svm.SVC(probability = True, random_state = 42)
        if model == "Random_Forest":
            pipeline = im_Pipeline([
                ("preprocessor", preprocessor),
                ("smote", SMOTE(sampling_strategy=1.0,random_state=42)),
                ("classifier", mdl)])
        else:
            pipeline = im_Pipeline([
                ("preprocessor", preprocessor),
                ("smote", SMOTE(sampling_strategy=1.0,random_state=42)),
                ("feature_selection", SelectKBest(score_func=f_classif)),
                ("classifier", mdl)])

        grid = GridSearchCV(
                estimator=pipeline,
                param_grid=param_grid[model],
                cv=cv_inner,
                scoring="f1_macro",
                n_jobs=-1)
        
        grid.fit(X_train, y_train, groups=groups_train)
        
        mdl_selected = grid.best_estimator_
        best_params_all[model].append(grid.best_params_)
        
        if model != "Random_Forest":
            selector = mdl_selected.named_steps["feature_selection"]
            selected_features = X_train.columns[selector.get_support()].tolist()
            print(selected_features)
        else:
            selected_features = X_train.columns.tolist()

        selected_features_all[model].append(selected_features)

        if model == "LogReg_Youden":
            y_train_proba_oof = cross_val_predict(
                mdl_selected,
                X_train,
                y_train,
                cv=cv_inner,   # cv_outer, pero aplicado a X_train, y_train, groups_train. no debes hacer es reutilizar outer_splits dentro del GridSearch, porque esos índices son del dataset completo, no de X_train.
                groups=groups_train,
                method="predict_proba",
                n_jobs=-1
            )[:, 1]

            fpr, tpr, thresholds = roc_curve(y_train, y_train_proba_oof)
            youden_index = tpr - fpr
            best_threshold = thresholds[np.argmax(youden_index)]

            thresholds_lr_youden.append(best_threshold)

            mdl_selected.fit(X_train, y_train)
            y_prob = mdl_selected.predict_proba(X_test)[:, 1]
            y_pred = (y_prob >= best_threshold).astype(int)
            
        elif model == "LogReg_Rho":
            y_train_proba_oof = cross_val_predict(mdl_selected, X_train, y_train, cv=cv_inner, groups=groups_train, method="predict_proba", n_jobs=-1)[:, 1]
            rho_threshold = get_best_rho(y_train,y_train_proba_oof, rhos=np.linspace(0.1, 0.9, 17))

            thresholds_lr_rho.append(rho_threshold)

            mdl_selected.fit(X_train, y_train)
            y_prob = mdl_selected.predict_proba(X_test)[:, 1]
            y_pred = (y_prob >= rho_threshold).astype(int)
        else:
            mdl_selected.fit(X_train, y_train)
            y_pred = mdl_selected.predict(X_test)
            y_prob = mdl_selected.predict_proba(X_test)[:, 1]
        
        accuracy[i, j] = accuracy_score(y_test, y_pred)
        balanced_accuracy[i, j] = balanced_accuracy_score(y_test, y_pred)
        f1_macro[i, j] = f1_score(y_test, y_pred, average="macro")
        f1_weighted[i, j] = f1_score(y_test, y_pred, average="weighted")
        auc[i, j] = roc_auc_score(y_test, y_prob)

        cm = confusion_matrix(y_test, y_pred)

        print(f"\n===== Fold {j+1} - {model} =====")
        print(f"Accuracy: {accuracy[i,j]:.4f}")
        print(f"Balanced Accuracy: {balanced_accuracy[i,j]:.4f}")
        print(f"F1 Macro: {f1_macro[i,j]:.4f}")
        print(f"AUC: {auc[i,j]:.4f}")

        print("\nConfusion matrix:")
        print(cm)

        print("\nClassification report:")
        print(classification_report(y_test, y_pred))
        
for (i,model) in enumerate(models):
    accuracy_model.append(f"{np.mean(accuracy[i,:]):.2f} +- {np.std(accuracy[i,:]):.2f} [{np.min(accuracy[i,:]):.2f} -  {np.max(accuracy[i,:]):.2f}]")
    balanced_accuracy_model.append(f"{np.mean(balanced_accuracy[i,:]):.2f} +- {np.std(balanced_accuracy[i,:]):.2f} [{np.min(balanced_accuracy[i,:]):.2f} -  {np.max(balanced_accuracy[i,:]):.2f}]")
    f1_macro_model.append(f"{np.mean(f1_macro[i,:]):.2f} +- {np.std(f1_macro[i,:]):.2f}  [{np.min(f1_macro[i,:]):.2f}  -   {np.max(f1_macro[i,:]):.2f}]")
    f1_weighted_model.append(f"{np.mean(f1_weighted[i,:]):.2f} +- {np.std(f1_weighted[i,:]):.2f}  [{np.min(f1_weighted[i,:]):.2f}  -   {np.max(f1_weighted[i,:]):.2f}]")
    AUC_model.append(f"{np.mean(auc[i,:]):.2f} +- {np.std(auc[i,:]):.2f}  [ {np.min(auc[i,:]):.2f}  -  {np.max(auc[i,:]):.2f}]")
        
    if i == 0:
        type_response_df.append("cramps")
    else:
        type_response_df.append("")
        
    model_df.append(model)


df_performance_models = pd.DataFrame(np.transpose([type_response_df,model_df,accuracy_model,balanced_accuracy_model,f1_macro_model,f1_weighted_model, AUC_model]), columns=["Type response","Model", "Accuracy","Balanced Accuracy","F1-Score (macro)","F1-Score (weighted)","AUC"])
display(df_performance_models)

['lh', 'estrogen', 'nightly_temperature', 'hr_mean', 'hr_median', 'hr_std', 'hr_cv', 'hr_var', 'hr_min', 'hr_max', 'hr_q_05', 'hr_q_25', 'hr_q_75', 'hr_q_95', 'glucose_mean', 'glucose_median', 'glucose_std', 'glucose_cv', 'glucose_var', 'glucose_min', 'glucose_max', 'glucose_q_05', 'glucose_q_25', 'glucose_q_75', 'glucose_q_95']

===== Fold 1 - KNN =====
Accuracy: 0.5487
Balanced Accuracy: 0.5273
F1 Macro: 0.5274
AUC: 0.5469

Confusion matrix:
[[43 25]
 [26 19]]

Classification report:
              precision    recall  f1-score   support

           0       0.62      0.63      0.63        68
           1       0.43      0.42      0.43        45

    accuracy                           0.55       113
   macro avg       0.53      0.53      0.53       113
weighted avg       0.55      0.55      0.55       113

['lh', 'nightly_temperature', 'hr_mean', 'hr_median', 'hr_cv', 'hr_min', 'hr_q_05', 'hr_q_25', 'hr_q_75', 'hr_q_95', 'glucose_mean', 'glucose_median', 'glucose_std', 'glucose_var', '

,Type response,Model,Accuracy,Balanced Accuracy,F1-Score (macro),F1-Score (weighted),AUC
0,cramps,KNN,0.48 +- 0.04 [0.43 - 0.55],0.47 +- 0.03 [0.45 - 0.53],0.46 +- 0.03 [0.42 - 0.53],0.49 +- 0.03 [0.45 - 0.55],0.47 +- 0.06 [ 0.38 - 0.55]
1,,LogReg_Youden,0.58 +- 0.03 [0.53 - 0.62],0.59 +- 0.03 [0.53 - 0.61],0.56 +- 0.03 [0.53 - 0.61],0.58 +- 0.03 [0.53 - 0.62],0.59 +- 0.05 [ 0.53 - 0.65]
2,,LogReg_Rho,0.57 +- 0.05 [0.49 - 0.64],0.58 +- 0.03 [0.53 - 0.63],0.56 +- 0.05 [0.49 - 0.63],0.57 +- 0.05 [0.48 - 0.63],0.59 +- 0.05 [ 0.53 - 0.65]
3,,Random_Forest,0.55 +- 0.05 [0.48 - 0.62],0.55 +- 0.03 [0.51 - 0.59],0.53 +- 0.04 [0.48 - 0.59],0.55 +- 0.05 [0.47 - 0.62],0.55 +- 0.05 [ 0.51 - 0.62]
4,,SVM,0.58 +- 0.04 [0.53 - 0.64],0.57 +- 0.05 [0.50 - 0.62],0.54 +- 0.05 [0.48 - 0.62],0.56 +- 0.05 [0.51 - 0.64],0.55 +- 0.10 [ 0.36 - 0.64]
